In [ ]:
pip install groq faiss-cpu sentence-transformers pdfplumber numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Phase 1: Load a text document

def load_txt(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    return text

# Test it — create a small sample file first
sample_text = """
Machine learning is a branch of artificial intelligence.
It allows computers to learn from data without being explicitly programmed.
There are three types: supervised, unsupervised, and reinforcement learning.
Neural networks are inspired by the human brain.
Deep learning uses many layers of neural networks.
"""

# Save sample to a file
with open('sample.txt', 'w') as f:
    f.write(sample_text)

# Now load it back
text = load_txt('sample.txt')
print("✅ File loaded successfully!")
print(f"📄 Total characters: {len(text)}")
print("\n--- Preview ---")
print(text[:200])

✅ File loaded successfully!
📄 Total characters: 311

--- Preview ---

Machine learning is a branch of artificial intelligence.
It allows computers to learn from data without being explicitly programmed.
There are three types: supervised, unsupervised, and reinforcement


In [ ]:
# Phase 2: Split text into overlapping chunks

def chunk_text(text, chunk_size=50, overlap=10):
    """
    Split text into overlapping word-based chunks.
    chunk_size = number of words per chunk
    overlap    = number of words shared between consecutive chunks
    """
    words = text.split()          # split into individual words
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end == len(words):     # reached the end
            break

        start += chunk_size - overlap  # move forward with overlap

    return chunks


# Test it on our sample text
chunks = chunk_text(text, chunk_size=30, overlap=5)

print(f"✅ Text split into {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk)
    print()

✅ Text split into 2 chunks

--- Chunk 1 ---
Machine learning is a branch of artificial intelligence. It allows computers to learn from data without being explicitly programmed. There are three types: supervised, unsupervised, and reinforcement learning. Neural networks

--- Chunk 2 ---
and reinforcement learning. Neural networks are inspired by the human brain. Deep learning uses many layers of neural networks.



In [ ]:
# Phase 3: Convert chunks into embeddings

from sentence_transformers import SentenceTransformer
import numpy as np

# Load the embedding model (downloads ~90MB first time, then cached)
print("⏳ Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded!\n")

# Convert all chunks into embeddings
embeddings = embedder.encode(chunks, convert_to_numpy=True).astype(np.float32)

# See what we got
print(f"📦 Number of chunks:     {len(chunks)}")
print(f"📐 Embedding dimensions: {embeddings.shape[1]}")
print(f"🔢 Embeddings shape:     {embeddings.shape}")
print(f"\n--- Preview of Chunk 1 embedding (first 8 numbers) ---")
print(embeddings[0][:8])

⏳ Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\ilang\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ilang\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded!

📦 Number of chunks:     2
📐 Embedding dimensions: 384
🔢 Embeddings shape:     (2, 384)

--- Preview of Chunk 1 embedding (first 8 numbers) ---
[-0.00283785 -0.05818539  0.0312207   0.00762109  0.02592722  0.00820744
 -0.03718356 -0.06078809]


In [ ]:
# Phase 4: Store embeddings in FAISS and search

import faiss
import numpy as np

# ── Step 1: Normalize embeddings (required for cosine similarity) ──
embeddings_copy = embeddings.copy()
faiss.normalize_L2(embeddings_copy)

# ── Step 2: Build the FAISS index ──
embedding_dim = embeddings_copy.shape[1]   # 384
index = faiss.IndexFlatIP(embedding_dim)   # IP = Inner Product = cosine similarity
index.add(embeddings_copy)                 # add all chunk embeddings

print(f"✅ FAISS index built!")
print(f"📦 Total chunks stored in index: {index.ntotal}\n")

# ── Step 3: Search with a question ──
question = "What are the types of machine learning?"

# Embed the question the same way we embedded chunks
question_embedding = embedder.encode([question], convert_to_numpy=True).astype(np.float32)
faiss.normalize_L2(question_embedding)

# Search top 2 most relevant chunks
top_k = 2
scores, indices = index.search(question_embedding, top_k)

print(f"🔍 Question: {question}\n")
print(f"📌 Top {top_k} most relevant chunks:\n")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
    print(f"  Rank {rank+1} (similarity score: {score:.4f})")
    print(f"  {chunks[idx]}")
    print()

✅ FAISS index built!
📦 Total chunks stored in index: 2

🔍 Question: What are the types of machine learning?

📌 Top 2 most relevant chunks:

  Rank 1 (similarity score: 0.7799)
  Machine learning is a branch of artificial intelligence. It allows computers to learn from data without being explicitly programmed. There are three types: supervised, unsupervised, and reinforcement learning. Neural networks

  Rank 2 (similarity score: 0.4803)
  and reinforcement learning. Neural networks are inspired by the human brain. Deep learning uses many layers of neural networks.



In [ ]:
# Phase 5: Send retrieved chunks to Groq LLM and get an answer

from groq import Groq

# ── Step 1: Paste your Groq API key here ──
GROQ_API_KEY = "your_key_here"

client = Groq(api_key=GROQ_API_KEY)

# ── Step 2: Build context from retrieved chunks ──
question = "What are the types of machine learning?"

# Retrieve top 2 relevant chunks (reusing from Phase 4)
question_embedding = embedder.encode([question], convert_to_numpy=True).astype(np.float32)
faiss.normalize_L2(question_embedding)
scores, indices = index.search(question_embedding, 2)

# Join the chunks into one context block
context = "\n\n".join([chunks[idx] for idx in indices[0]])

print("📋 Context being sent to LLM:")
print(context)
print("\n" + "─"*50 + "\n")

# ── Step 3: Build the prompt ──
prompt = f"""You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know based on the documents."

Context:
{context}

Question: {question}

Answer:"""

# ── Step 4: Call the LLM ──
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0.2,
    max_tokens=512
)

answer = response.choices[0].message.content.strip()

print(f"🙋 Question: {question}")
print(f"\n🤖 Answer:\n{answer}")

📋 Context being sent to LLM:
Machine learning is a branch of artificial intelligence. It allows computers to learn from data without being explicitly programmed. There are three types: supervised, unsupervised, and reinforcement learning. Neural networks

and reinforcement learning. Neural networks are inspired by the human brain. Deep learning uses many layers of neural networks.

──────────────────────────────────────────────────

🙋 Question: What are the types of machine learning?

🤖 Answer:
The three types of machine learning are: supervised, unsupervised, and reinforcement learning.


In [ ]:
# Phase 6: Multi-turn conversation with chat history

def ask(question, chat_history=[]):
    # ── Step 1: Retrieve relevant chunks ──
    q_embedding = embedder.encode([question], convert_to_numpy=True).astype(np.float32)
    faiss.normalize_L2(q_embedding)
    scores, indices = index.search(q_embedding, 2)
    context = "\n\n".join([chunks[idx] for idx in indices[0]])

    # ── Step 2: Build messages with history ──
    system_message = {
        "role": "system",
        "content": """You are a helpful assistant. Answer questions using ONLY 
        the provided context. If the answer is not in the context, 
        say 'I don't know based on the documents.'"""
    }

    # Format context as first user message
    context_message = {
        "role": "user",
        "content": f"Here is the context from the documents:\n\n{context}"
    }
    context_ack = {
        "role": "assistant", 
        "content": "Understood! I'll answer based on this context."
    }

    # Build full message list: system + context + history + new question
    messages = [system_message, context_message, context_ack]
    messages += chat_history  # add previous turns
    messages.append({"role": "user", "content": question})

    # ── Step 3: Call LLM ──
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        temperature=0.2,
        max_tokens=512
    )

    answer = response.choices[0].message.content.strip()

    # ── Step 4: Save this turn to history ──
    chat_history.append({"role": "user", "content": question})
    chat_history.append({"role": "assistant", "content": answer})

    return answer, chat_history


# ── Test multi-turn conversation ──
history = []

# Turn 1
q1 = "What are the types of machine learning?"
answer1, history = ask(q1, history)
print(f"You: {q1}")
print(f"AI:  {answer1}\n")

# Turn 2 — follow-up question referencing previous answer
q2 = "Can you explain the first one in more detail?"
answer2, history = ask(q2, history)
print(f"You: {q2}")
print(f"AI:  {answer2}\n")

# Turn 3 — another follow-up
q3 = "What about neural networks?"
answer3, history = ask(q3, history)
print(f"You: {q3}")
print(f"AI:  {answer3}\n")

print(f"📜 Total conversation turns stored: {len(history)}")

You: What are the types of machine learning?
AI:  There are three types of machine learning: 
1. Supervised learning
2. Unsupervised learning
3. Reinforcement learning.

You: Can you explain the first one in more detail?
AI:  I don't know based on the documents.

You: What about neural networks?
AI:  Neural networks are inspired by the human brain. Deep learning uses many layers of neural networks.

📜 Total conversation turns stored: 6
